In [2]:
import pandas as pd
import numpy as np

In [3]:
accounts = pd.read_csv('accounts.csv')
products = pd.read_csv('products.csv')
pipeline = pd.read_csv('sales_pipeline.csv')
teams = pd.read_csv('sales_teams.csv')

In [4]:
print(accounts.columns)
print(products.columns)
print(pipeline.columns)
print(teams.columns)

Index(['account', 'sector', 'year_established', 'revenue', 'employees',
       'office_location', 'subsidiary_of'],
      dtype='object')
Index(['product', 'series', 'sales_price'], dtype='object')
Index(['opportunity_id', 'sales_agent', 'product', 'account', 'deal_stage',
       'engage_date', 'close_date', 'close_value'],
      dtype='object')
Index(['sales_agent', 'manager', 'regional_office'], dtype='object')


In [5]:
accounts.drop_duplicates(inplace=True)
products.drop_duplicates(inplace=True)
pipeline.drop_duplicates(inplace=True)
teams.drop_duplicates(inplace=True)

In [6]:
pipeline['close_value'] = pipeline['close_value'].fillna(0)

accounts.fillna('Unknown', inplace=True)

In [7]:
pipeline['close_date'] = pd.to_datetime(pipeline['close_date'], errors='coerce')

In [8]:
accounts['sector'] = accounts['sector'].str.lower().str.strip()

In [9]:
# Merge accounts
master = pipeline.merge(accounts, on='account', how='left')

# Merge products
master = master.merge(products, on='product', how='left')

# Merge sales teams
master = master.merge(teams, on='sales_agent', how='left')

In [10]:
print(master.shape)
master.head()

(8800, 18)


,opportunity_id,sales_agent,product,account,deal_stage,engage_date,close_date,close_value,sector,year_established,revenue,employees,office_location,subsidiary_of,series,sales_price,manager,regional_office
0,1C1I7A6R,Moses Frase,GTX Plus Basic,Cancity,Won,2016-10-20,2017-03-01,1054.0,retail,2001.0,718.62,2448.0,United States,Unknown,GTX,1096.0,Dustin Brinkmann,Central
1,Z063OYW0,Darcel Schlecht,GTXPro,Isdom,Won,2016-10-25,2017-03-11,4514.0,medical,2002.0,3178.24,4540.0,United States,Unknown,NaN,NaN,Melvin Marxen,Central
2,EC4QE1BX,Darcel Schlecht,MG Special,Cancity,Won,2016-10-25,2017-03-07,50.0,retail,2001.0,718.62,2448.0,United States,Unknown,MG,55.0,Melvin Marxen,Central
3,MV1LWRNH,Moses Frase,GTX Basic,Codehow,Won,2016-10-25,2017-03-09,588.0,software,1998.0,2714.90,2641.0,United States,Acme Corporation,GTX,550.0,Dustin Brinkmann,Central
4,PE84CX4O,Zane Levy,GTX Basic,Hatfan,Won,2016-10-25,2017-03-02,517.0,services,1982.0,792.46,1299.0,United States,Unknown,GTX,550.0,Summer Sewald,West


In [11]:
# Win flag
master['is_won'] = master['deal_stage'].apply(lambda x: 1 if x == 'Won' else 0)

# Deal size
master['deal_size'] = pd.cut(master['close_value'],
                             bins=[0, 1000, 5000, 100000],
                             labels=['small', 'medium', 'large'])

# Revenue per employee (advanced insight)
master['rev_per_employee'] = master['revenue'] / master['employees']

In [12]:
stage_counts = master['deal_stage'].value_counts()
print(stage_counts)

conversion_rate = master['is_won'].mean()
print("Win Rate:", conversion_rate)

deal_stage
Won            4238
Lost           2473
Engaging       1589
Prospecting     500
Name: count, dtype: int64
Win Rate: 0.48159090909090907


In [13]:
sector_perf = master.groupby('sector')['close_value'].sum().sort_values(ascending=False)
print(sector_perf)

sector
retail                1867528.0
technolgy             1515487.0
medical               1359595.0
software              1077934.0
finance                950908.0
marketing              922321.0
entertainment          689007.0
telecommunications     653574.0
services               533006.0
employment             436174.0
Name: close_value, dtype: float64


In [14]:
sales_perf = master.groupby('manager').agg({
    'opportunity_id': 'count',
    'is_won': 'mean',
    'close_value': 'sum'
}).rename(columns={
    'opportunity_id': 'total_deals',
    'is_won': 'win_rate'
})

print(sales_perf.sort_values(by='win_rate', ascending=False))

                  total_deals  win_rate  close_value
manager                                             
Rocco Neubert            1327  0.520723    1960545.0
Cara Losch                964  0.497925    1130049.0
Summer Sewald            1701  0.486772    1964750.0
Dustin Brinkmann         1583  0.471889    1094363.0
Celia Rouche             1296  0.470679    1603897.0
Melvin Marxen            1929  0.457232    2251930.0


In [15]:
product_perf = master.groupby('product').agg({
    'close_value': 'sum',
    'is_won': 'mean'
})

print(product_perf.sort_values(by='close_value', ascending=False))

                close_value    is_won
product                              
GTXPro            3510578.0  0.492568
GTX Plus Pro      2629651.0  0.494835
MG Advanced       2216387.0  0.463173
GTX Plus Basic     705275.0  0.472162
GTX Basic          499263.0  0.490354
GTK 500            400612.0  0.375000
MG Special          43768.0  0.480315


In [16]:
print(master.isnull().sum())

opportunity_id         0
sales_agent            0
product                0
account             1425
deal_stage             0
engage_date          500
close_date          2089
close_value            0
sector              1425
year_established    1425
revenue             1425
employees           1425
office_location     1425
subsidiary_of       1425
series              1480
sales_price         1480
manager                0
regional_office        0
is_won                 0
deal_size           4562
rev_per_employee    1425
dtype: int64


In [17]:
master.to_csv('cleaned_master_dataset.csv', index=False)

from google.colab import files
files.download('cleaned_master_dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>